In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
import pandas as pd
import matplotlib.cm as cm
import os


import warnings
warnings.filterwarnings('ignore')


In [8]:
# Load your pickle file
exp_name = '20241110_1'
pickle_path = '/run/user/1000/gvfs/smb-share:server=bucket.oist.jp,share=bucket/ReiterU/Anouk/basler_data/'+ exp_name+ '/'
output_folder_path = '/run/user/1000/gvfs/smb-share:server=bucket.oist.jp,share=bucket/ReiterU/Anouk/basler_data/'+ exp_name+ '/'


# pickle_path = '/Volumes/ReiterU/Anouk/basler_data/'+ exp_name+ '/'
# output_folder_path = '/Volumes/ReiterU/Anouk/basler_data/'+ exp_name+ '/'

pickle_name_col1 = exp_name + '_1_aruco_cleaned_merged_single_track_col1.pkl'
pickle_name_col2 = exp_name + '_1_aruco_cleaned_merged_single_track_col2.pkl'

In [9]:
def read_pickle_file_with_load(pickle_path, pickle_name):
    """
    Read a DataFrame from a pickle file using `pickle.load`, ensuring the file exists.

    Parameters:
    - pickle_path (str): Path to the folder where the pickle file is located.
    - pickle_name (str): Name of the pickle file (including '.pkl').

    Returns:
    - pd.DataFrame: Loaded DataFrame.
    """
    # Full path to the pickle file
    pickle_full_path = os.path.join(pickle_path, pickle_name)
    
    try:
        print(f"Reading DataFrame from {pickle_full_path} using `pickle.load`...")
        with open(pickle_full_path, 'rb') as f:
            data = pickle.load(f)
        print(f"File '{pickle_name}' read successfully.")
        return data
    except FileNotFoundError:
        print(f"Error: File '{pickle_full_path}' not found.")
    except Exception as e:
        print(f"Error while reading file: {e}")


In [10]:
df_1 = read_pickle_file_with_load(output_folder_path,  pickle_name_col1)
df_2 = read_pickle_file_with_load(output_folder_path, pickle_name_col2)



Reading DataFrame from /run/user/1000/gvfs/smb-share:server=bucket.oist.jp,share=bucket/ReiterU/Anouk/basler_data/20241110_1/20241110_1_1_aruco_cleaned_merged_single_track_col1.pkl using `pickle.load`...
Error while reading file: No module named 'numpy._core.numeric'
Reading DataFrame from /run/user/1000/gvfs/smb-share:server=bucket.oist.jp,share=bucket/ReiterU/Anouk/basler_data/20241110_1/20241110_1_1_aruco_cleaned_merged_single_track_col2.pkl using `pickle.load`...
Error while reading file: No module named 'numpy._core.numeric'


In [ ]:
def get_speed_and_acc(df):
    df['X_shift'] = df.groupby('ARUCO_number')['X'].shift(-1)
    df['Y_shift'] = df.groupby('ARUCO_number')['Y'].shift(-1)
    df['Frame_shift'] = df.groupby('ARUCO_number')['Frame_number'].shift(-1)
    
    # Calculate distance between frames
    df['distance'] = np.sqrt((df['X_shift'] - df['X'])**2 + (df['Y_shift'] - df['Y'])**2)
    
    # Calculate time interval between frames
    df['time_interval'] = df['Frame_shift'] - df['Frame_number']
    
    # Speed (distance per frame or second, depending on your frame rate)
    df['speed'] = df['distance'] / df['time_interval']
    
    # Shift speed to compute acceleration
    df['speed_shift'] = df.groupby('ARUCO_number')['speed'].shift(-1)
    df['acceleration'] = (df['speed_shift'] - df['speed']) / df['time_interval']
    
    # Drop intermediate columns used for calculation
    df = df.drop(columns=['X_shift', 'Y_shift', 'Frame_shift', 'speed_shift', 'time_interval'])
    
    return df 
        

In [ ]:
df_1_with_speed = get_speed_and_acc(df_1)
df_2_with_speed = get_speed_and_acc(df_2)


In [ ]:
plt.scatter(df_1['X'], df_1['Y'], s = 1)
plt.show()
plt.scatter(df_2['X'], df_2['Y'], s = 1)
plt.show()

In [ ]:
#Define a cutoff for speed 

#Original speed and acc distributions
def plot_speed_and_acc(df_1, df_2):
    # Create a figure with subplots
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    
    # Plot speed for colony 1 (df_1)
    axes[0, 0].hist(df_1['speed'], bins=30, log=True, color='blue', alpha=0.7)
    axes[0, 0].set_title("Colony 1: Speed")
    axes[0, 0].set_xlabel("Speed")
    axes[0, 0].set_ylabel("Frequency (log scale)")
    
    # Plot speed for colony 2 (df_2)
    axes[0, 1].hist(df_2['speed'], bins=30, log=True, color='orange', alpha=0.7)
    axes[0, 1].set_title("Colony 2: Speed")
    axes[0, 1].set_xlabel("Speed")
    axes[0, 1].set_ylabel("Frequency (log scale)")
    
    # Plot acceleration for colony 1 (df_1)
    axes[1, 0].hist(df_1['acceleration'], bins=30, log=True, color='green', alpha=0.7)
    axes[1, 0].set_title("Colony 1: Acceleration")
    axes[1, 0].set_xlabel("Acceleration")
    axes[1, 0].set_ylabel("Frequency (log scale)")
    
    # Plot acceleration for colony 2 (df_2)
    axes[1, 1].hist(df_2['acceleration'], bins=30, log=True, color='red', alpha=0.7)
    axes[1, 1].set_title("Colony 2: Acceleration")
    axes[1, 1].set_xlabel("Acceleration")
    axes[1, 1].set_ylabel("Frequency (log scale)")
    
    # Adjust layout
    plt.tight_layout()
    plt.show()
    return 
plot_speed_and_acc(df_1_with_speed, df_2_with_speed)

In [ ]:
#Choose a threshold to cleanout the speeds and accelerations 
thresh = 150
df_1_cleaned = df_1_with_speed[df_1_with_speed['speed']< thresh]
df_2_cleaned = df_2_with_speed[df_2_with_speed['speed']< thresh]

plot_speed_and_acc(df_1_cleaned, df_2_cleaned)

In [ ]:
def check_tracks(df):

    # Define a colormap for speed
    colormap = cm.viridis
    
    # Loop over each ARUCO tag to create an individual plot
    for aruco_id, data in df.groupby('ARUCO_number'):
        fig, ax = plt.subplots(figsize=(10, 8))
        ax.set_xlabel('X')
        ax.set_ylabel('Y')
        ax.set_title(f'Trajectory of ARUCO {int(aruco_id)} Colored by Speed')
    
        # Normalize speed for the current ARUCO's range
        norm = plt.Normalize(data['speed'].min(), data['speed'].max())
        
        # Scatter plot for each ARUCO's trajectory, colored by speed
        scatter = ax.scatter(data['X'], data['Y'], c=colormap(norm(data['speed'])), s=5, alpha=0.6)
    
        # Add a color bar for speed
        cbar = fig.colorbar(cm.ScalarMappable(norm=norm, cmap=colormap), ax=ax, label='Speed')
    
        # Display the plot for the current ARUCO tag
        plt.show()

    return

#check_tracks(df_2_cleaned[df_2_cleaned['ARUCO_number'] == 202])
#check_tracks(df_2_cleaned)

In [ ]:
#Save the active_arucos (uncleaned) 
def convert_to_pickle(folder_path, pickle_name, df_to_convert): 
    #Create a new folder with the combined files in the desired directory 
    folder_name = folder_path.split('/')[-2]
    if not os.path.exists(folder_path):
        # If the folder does not exist, create it
        os.makedirs(folder_path)
        print(f"Folder '{folder_name}' created in '{folder_path}'.")
    else:
        print(f"Folder '{folder_name}' already exists in '{folder_path}'.")

    df_to_convert.to_pickle(folder_path +  pickle_name )
    
    return

convert_to_pickle(output_folder_path, exp_name + '_2_aruco_cleaned_with_speed_col1.pkl', df_1_cleaned)
convert_to_pickle(output_folder_path, exp_name + '_2_aruco_cleaned_with_speed_col2.pkl', df_2_cleaned)
